In [1]:
from pathlib import Path
import sys

# Add src to import path
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_PATH = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_PATH))

In [14]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

from pydantic import BaseModel, Field
import json
import re

from kg import CustomRAKG
from rst import DMRSTParser

In [3]:
s1 = 'Google released Gemini 2.5 to improve reasoning and multimodal capabilities across its AI products.'
s2 = 'Several months later, Google integrated Gemini 2.5 into Google Workspace, allowing users to generate documents, summarize emails, and analyze spreadsheets.'

In [4]:
# load LLM models
llm = ChatOpenAI(
    base_url='http://127.0.0.1:1234/v1',
    model='google/gemma-4-e2b',
    api_key='none',
    temperature=0,
    extra_body={
        'reasoning' : {
            'effort' : 'low'
        }
    }
)

embed = OpenAIEmbeddings(
    base_url='http://127.0.0.1:1234/v1',
    model='text-embedding-bge-m3',
    api_key='none',
    check_embedding_ctx_length=False
)

In [5]:
kg = CustomRAKG(llm, embed)
rst = DMRSTParser(checkpoint_path='../artifacts/dmrst/multi_all_checkpoint.torchsave', cache_dir='../artifacts/models')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 313.79it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ignoring old checkpoint key: encoder.language_model.embeddings.position_ids


In [7]:
out_kg = kg.get_kg_from_sents([s1, s2])
final_kg = kg.convert_kg(out_kg)

In [10]:
final_rst = rst.parse('\n'.join([s1, s2]))

In [13]:
final_rst

{'text': 'Google released Gemini 2.5 to improve reasoning and multimodal capabilities across its AI products.\nSeveral months later, Google integrated Gemini 2.5 into Google Workspace, allowing users to generate documents, summarize emails, and analyze spreadsheets.',
 'tokens': ['▁Google',
  '▁released',
  '▁Ge',
  'mini',
  '▁2.5',
  '▁to',
  '▁improve',
  '▁reason',
  'ing',
  '▁and',
  '▁multi',
  'mo',
  'dal',
  '▁capabil',
  'ities',
  '▁across',
  '▁its',
  '▁AI',
  '▁products',
  '.',
  '▁Sever',
  'al',
  '▁months',
  '▁later',
  ',',
  '▁Google',
  '▁integrat',
  'ed',
  '▁Ge',
  'mini',
  '▁2.5',
  '▁into',
  '▁Google',
  '▁Work',
  'space',
  ',',
  '▁',
  'allowing',
  '▁users',
  '▁to',
  '▁generate',
  '▁documents',
  ',',
  '▁summa',
  'riz',
  'e',
  '▁email',
  's',
  ',',
  '▁and',
  '▁anal',
  'y',
  'ze',
  '▁spread',
  'she',
  'ets',
  '.'],
 'edu_breaks': [4, 19, 35, 56],
 'edus': ['Google released Gemini 2.5',
  'to improve reasoning and multimodal capabilitie

In [12]:
print(json.dumps(
    final_kg,
    ensure_ascii=False,
    indent=2,
    default=str
))

{
  "entities": [
    {
      "name": "Gemini 2.5",
      "type": "AI Model",
      "description": "An AI model released by Google.",
      "description_source_sentences": [
        0
      ],
      "attributes": {
        "release_source": {
          "value": "Google",
          "source_sentences": [
            0
          ]
        },
        "purpose": {
          "value": "improve reasoning and multimodal capabilities",
          "source_sentences": [
            0
          ]
        }
      }
    },
    {
      "name": "Google",
      "type": "Organization",
      "description": "A company that releases AI products and integrates them into various services.",
      "description_source_sentences": [
        0,
        1
      ],
      "attributes": {
        "released_product": {
          "value": "Gemini 2.5",
          "source_sentences": [
            0
          ]
        },
        "integrated_product": {
          "value": "Gemini 2.5",
          "source_sentences": [
   

In [ ]:
class ItemEduAlignment(BaseModel):
    item_id: str
    edu_ids: list[int]

class RSTEdgeSelection(BaseModel):
    source_item_id: str | None
    target_item_id: str | None

# Prompts

def get_alignment_prompt() -> str:
    return '''
You are aligning one knowledge graph item with Elementary Discourse Units (EDUs).

Select the EDU IDs that directly express the provided knowledge graph item.

Rules:
1. Use only the exact EDU IDs provided in Candidate EDUs.
2. Select only the EDU or EDUs that directly state the item.
3. Do not include EDUs that only provide unnecessary context.
4. Select the smallest possible set of EDUs.
5. Do not create, rewrite, rename, or correct the item.
6. For a relation, consider its source, relation, and target.
7. For an attribute, consider its entity, attribute name, and value.
8. If none of the candidate EDUs supports the item, return an empty list.
9. Preserve the item ID exactly as provided.
10. Do not return EDU IDs that are not explicitly provided.

Item:
{item}

Candidate EDUs:
{edus}
'''.strip()

def get_rst_edge_prompt() -> str:
    return '''
Select one knowledge graph item from the left candidates and one knowledge graph item from the right candidates that best represent the RST relation.

Rules:
1. Select only IDs from the provided candidates.
2. The source item must come from Left candidates.
3. The target item must come from Right candidates.
4. Prefer relations over attributes when both represent the same proposition.
5. Select the item that best represents the complete meaning of each text span.
6. Do not create, rewrite, rename, or correct item IDs.
7. Return null if no suitable item exists on one side.
8. Select only one item from each side.

RST relation:
{relation}

Left text:
{left_text}

Right text:
{right_text}

Left candidates:
{left_candidates}

Right candidates:
{right_candidates}
'''.strip()

def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([,.!?;:])', r'\1', text)

    return text.strip()

def remove_punctuation(text: str) -> str:
    return re.sub(
        r'[,.!?;:]',
        '',
        normalize_text(text),
    )

def align_sentences_to_edus(
    sentences: list[str],
    edus: list[str],
) -> dict[int, list[int]]:
    sentence_to_edus = {}
    edu_cursor = 0

    for sentence_id, sentence in enumerate(sentences):
        target = remove_punctuation(sentence)

        accumulated = ''
        matched_edu_ids = []

        while edu_cursor < len(edus):
            edu_id = edu_cursor

            accumulated = normalize_text(
                f'{accumulated} {edus[edu_cursor]}'
            )

            matched_edu_ids.append(edu_id)
            edu_cursor += 1

            current = remove_punctuation(accumulated)

            if current == target:
                break

            if not target.startswith(current):
                raise ValueError(
                    f'Could not align sentence {sentence_id} '
                    f'with the EDU sequence. '
                    f'Current text: {current!r}'
                )

        if remove_punctuation(accumulated) != target:
            raise ValueError(
                f'Incomplete EDU alignment for sentence '
                f'{sentence_id}.'
            )

        sentence_to_edus[sentence_id] = matched_edu_ids

    if edu_cursor != len(edus):
        raise ValueError(
            'Some EDUs were not aligned with any sentence.'
        )

    return sentence_to_edus

def extract_kg_items(kg: dict) -> list[dict]:
    items = []

    for index, relation in enumerate(
        kg.get('relations', [])
    ):
        items.append({
            'id': f'relation_{index}',
            'type': 'relation',
            'source': relation.get('source'),
            'relation': relation.get('relation'),
            'target': relation.get('target'),
            'source_sentences': relation.get(
                'source_sentences',
                [],
            ),
        })

    for entity_index, entity in enumerate(
        kg.get('entities', [])
    ):
        for attribute_name, attribute in entity.get(
            'attributes',
            {},
        ).items():
            items.append({
                'id': (
                    f'entity_{entity_index}_'
                    f'attribute_{attribute_name}'
                ),
                'type': 'attribute',
                'entity': entity.get('name'),
                'attribute': attribute_name,
                'value': attribute.get('value'),
                'source_sentences': attribute.get(
                    'source_sentences',
                    [],
                ),
            })

    return items

def get_candidate_edus(
    item: dict,
    sentence_to_edus: dict[int, list[int]],
    edus: list[str],
) -> list[dict]:
    candidate_ids = sorted({
        edu_id
        for sentence_id in item.get(
            'source_sentences',
            [],
        )
        for edu_id in sentence_to_edus.get(
            sentence_id,
            [],
        )
    })

    return [
        {
            'edu_id': edu_id,
            'text': edus[edu_id],
        }
        for edu_id in candidate_ids
    ]

def align_items_to_edus(
    llm,
    items: list[dict],
    edus: list[str],
    sentence_to_edus: dict[int, list[int]],
) -> dict[str, list[int]]:
    prompt = ChatPromptTemplate.from_template(
        get_alignment_prompt()
    )

    structured_llm = llm.with_structured_output(
        ItemEduAlignment
    )

    chain = prompt | structured_llm

    result = {}

    for item in items:
        candidate_edus = get_candidate_edus(
            item=item,
            sentence_to_edus=sentence_to_edus,
            edus=edus,
        )

        if not candidate_edus:
            result[item['id']] = []
            continue

        response = chain.invoke({
            'item': json.dumps(
                item,
                ensure_ascii=False,
                indent=2,
                default=str,
            ),
            'edus': json.dumps(
                candidate_edus,
                ensure_ascii=False,
                indent=2,
                default=str,
            ),
        })

        if response.item_id != item['id']:
            raise ValueError(
                f'Invalid item ID returned by the LLM. '
                f'Expected {item["id"]!r}, '
                f'received {response.item_id!r}.'
            )

        valid_edu_ids = {
            edu['edu_id']
            for edu in candidate_edus
        }

        invalid_edu_ids = (
            set(response.edu_ids)
            - valid_edu_ids
        )

        if invalid_edu_ids:
            raise ValueError(
                f'Invalid EDU IDs returned for '
                f'{item["id"]}: '
                f'{sorted(invalid_edu_ids)}. '
                f'Valid IDs: {sorted(valid_edu_ids)}'
            )

        result[item['id']] = sorted(
            set(response.edu_ids)
        )

    return result

_RST_NODE_PATTERN = re.compile(
    r'''
    \(
        (?P<left_start>\d+):
        (?P<left_role>Nucleus|Satellite)=
        (?P<left_relation>[^:,\s]+):
        (?P<left_end>\d+),
        (?P<right_start>\d+):
        (?P<right_role>Nucleus|Satellite)=
        (?P<right_relation>[^:,\s]+):
        (?P<right_end>\d+)
    \)
    ''',
    re.VERBOSE,
)

def parse_rst_tree(tree: str) -> list[dict]:
    nodes = []

    for index, match in enumerate(
        _RST_NODE_PATTERN.finditer(tree)
    ):
        data = match.groupdict()

        left_role = data['left_role']
        right_role = data['right_role']

        nuclearity = (
            ('N' if left_role == 'Nucleus' else 'S')
            + ('N' if right_role == 'Nucleus' else 'S')
        )

        left_relation = data['left_relation']
        right_relation = data['right_relation']

        if left_relation != 'span':
            relation = left_relation

        elif right_relation != 'span':
            relation = right_relation

        else:
            relation = 'span'

        original_left_span = [
            int(data['left_start']),
            int(data['left_end']),
        ]

        original_right_span = [
            int(data['right_start']),
            int(data['right_end']),
        ]

        nodes.append({
            'id': f'rst_{index}',
            'left_span': [
                original_left_span[0] - 1,
                original_left_span[1] - 1,
            ],
            'right_span': [
                original_right_span[0] - 1,
                original_right_span[1] - 1,
            ],
            'original_left_span': original_left_span,
            'original_right_span': original_right_span,
            'relation': relation,
            'nuclearity': nuclearity,
            'left_role': left_role,
            'right_role': right_role,
        })

    return nodes

def get_items_in_span(
    item_edus: dict[str, list[int]],
    span: list[int],
) -> list[str]:
    start, end = span

    return [
        item_id
        for item_id, edu_ids in item_edus.items()
        if any(
            start <= edu_id <= end
            for edu_id in edu_ids
        )
    ]


def get_edus_in_span(
    edus: list[str],
    span: list[int],
) -> list[dict]:
    start, end = span

    return [
        {
            'edu_id': edu_id,
            'text': edus[edu_id],
        }
        for edu_id in range(start, end + 1)
    ]


def join_span_text(
    edus: list[str],
    span: list[int],
) -> str:
    start, end = span

    return ' '.join(
        edus[edu_id]
        for edu_id in range(start, end + 1)
    )

def select_rst_edge_items(
    llm,
    node: dict,
    left_item_ids: list[str],
    right_item_ids: list[str],
    items_by_id: dict[str, dict],
    edus: list[str],
) -> tuple[str | None, str | None]:
    if not left_item_ids or not right_item_ids:
        return None, None

    if (
        len(left_item_ids) == 1
        and len(right_item_ids) == 1
    ):
        return left_item_ids[0], right_item_ids[0]

    prompt = ChatPromptTemplate.from_template(
        get_rst_edge_prompt()
    )

    structured_llm = llm.with_structured_output(
        RSTEdgeSelection
    )

    chain = prompt | structured_llm

    left_candidates = [
        items_by_id[item_id]
        for item_id in left_item_ids
    ]

    right_candidates = [
        items_by_id[item_id]
        for item_id in right_item_ids
    ]

    response = chain.invoke({
        'relation': node['relation'],
        'left_text': join_span_text(
            edus=edus,
            span=node['left_span'],
        ),
        'right_text': join_span_text(
            edus=edus,
            span=node['right_span'],
        ),
        'left_candidates': json.dumps(
            left_candidates,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        'right_candidates': json.dumps(
            right_candidates,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
    })

    if (
        response.source_item_id is not None
        and response.source_item_id not in left_item_ids
    ):
        raise ValueError(
            f'Invalid source item returned by the LLM: '
            f'{response.source_item_id}'
        )

    if (
        response.target_item_id is not None
        and response.target_item_id not in right_item_ids
    ):
        raise ValueError(
            f'Invalid target item returned by the LLM: '
            f'{response.target_item_id}'
        )

    return (
        response.source_item_id,
        response.target_item_id,
    )

def build_rst_edges(
    llm,
    rst_nodes: list[dict],
    item_edus: dict[str, list[int]],
    items: list[dict],
    edus: list[str],
) -> list[dict]:
    items_by_id = {
        item['id']: item
        for item in items
    }

    edges = []

    for node in rst_nodes:
        left_item_ids = get_items_in_span(
            item_edus=item_edus,
            span=node['left_span'],
        )

        right_item_ids = get_items_in_span(
            item_edus=item_edus,
            span=node['right_span'],
        )

        source_item_id, target_item_id = (
            select_rst_edge_items(
                llm=llm,
                node=node,
                left_item_ids=left_item_ids,
                right_item_ids=right_item_ids,
                items_by_id=items_by_id,
                edus=edus,
            )
        )

        if source_item_id is None:
            continue

        if target_item_id is None:
            continue

        if source_item_id == target_item_id:
            continue

        edges.append({
            'source_item': source_item_id,
            'target_item': target_item_id,
            'relation': node['relation'],
            'nuclearity': node['nuclearity'],
            'source_role': node['left_role'],
            'target_role': node['right_role'],
            'source_span': node['left_span'],
            'target_span': node['right_span'],
            'rst_node_id': node['id'],
        })

    return edges

def enrich_kg_with_rst(
    kg: dict,
    rst_output: dict,
    sentences: list[str],
    llm,
) -> dict:
    edus = rst_output['edus']
    tree = rst_output['tree']

    sentence_to_edus = align_sentences_to_edus(
        sentences=sentences,
        edus=edus,
    )

    items = extract_kg_items(kg)

    item_edus = align_items_to_edus(
        llm=llm,
        items=items,
        edus=edus,
        sentence_to_edus=sentence_to_edus,
    )

    rst_nodes = parse_rst_tree(tree)

    rst_edges = build_rst_edges(
        llm=llm,
        rst_nodes=rst_nodes,
        item_edus=item_edus,
        items=items,
        edus=edus,
    )

    return {
        'knowledge_graph': kg,
        'rst_enrichment': {
            'sentence_to_edus': sentence_to_edus,
            'items': items,
            'item_edus': item_edus,
            'rst_nodes': rst_nodes,
            'rst_edges': rst_edges,
        },
    }

In [54]:
rst_kg = enrich_kg_with_rst(final_kg, final_rst, [s1, s2], llm)

In [55]:
print(json.dumps(
    rst_kg,
    ensure_ascii=False,
    indent=2,
    default=str
))

{
  "knowledge_graph": {
    "entities": [
      {
        "name": "Gemini 2.5",
        "type": "AI Model",
        "description": "An AI model released by Google.",
        "description_source_sentences": [
          0
        ],
        "attributes": {
          "release_source": {
            "value": "Google",
            "source_sentences": [
              0
            ]
          },
          "purpose": {
            "value": "improve reasoning and multimodal capabilities",
            "source_sentences": [
              0
            ]
          }
        }
      },
      {
        "name": "Google",
        "type": "Organization",
        "description": "A company that releases AI products and integrates them into various services.",
        "description_source_sentences": [
          0,
          1
        ],
        "attributes": {
          "released_product": {
            "value": "Gemini 2.5",
            "source_sentences": [
              0
            ]
          },
 

In [56]:
def get_item_label(item: dict) -> str:
    if item['type'] == 'relation':
        relation = item['relation'].replace('_', ' ')

        return (
            f'{item["source"]} '
            f'--{relation}--> '
            f'{item["target"]}'
        )

    if item['type'] == 'attribute':
        attribute = item['attribute'].replace('_', ' ')

        return (
            f'{item["entity"]} '
            f'--{attribute}--> '
            f'{item["value"]}'
        )

    return item['id']


def simplify_rst_enrichment(
    result: dict,
    rst_output: dict,
) -> dict:
    enrichment = result['rst_enrichment']
    edus = rst_output['edus']

    items_by_id = {
        item['id']: item
        for item in enrichment['items']
    }

    simplified_edus = [
        {
            'id': edu_id,
            'text': edu,
        }
        for edu_id, edu in enumerate(edus)
    ]

    simplified_items = []

    for item_id, edu_ids in enrichment['item_edus'].items():
        item = items_by_id[item_id]

        simplified_items.append({
            'id': item_id,
            'label': get_item_label(item),
            'edu_ids': edu_ids,
            'edu_texts': [
                edus[edu_id]
                for edu_id in edu_ids
            ],
        })

    simplified_edges = []

    for edge in enrichment['rst_edges']:
        source_item = items_by_id[edge['source_item']]
        target_item = items_by_id[edge['target_item']]

        simplified_edges.append({
            'source': get_item_label(source_item),
            'relation': edge['relation'],
            'target': get_item_label(target_item),
            'nuclearity': edge['nuclearity'],
            'source_role': edge['source_role'],
            'target_role': edge['target_role'],
        })

    return {
        'edus': simplified_edus,
        'items': simplified_items,
        'rst_edges': simplified_edges,
    }

In [57]:
simplified = simplify_rst_enrichment(
    result=rst_kg,
    rst_output=final_rst,
)

print(
    json.dumps(
        simplified,
        ensure_ascii=False,
        indent=2,
        default=str,
    )
)

{
  "edus": [
    {
      "id": 0,
      "text": "Google released Gemini 2.5"
    },
    {
      "id": 1,
      "text": "to improve reasoning and multimodal capabilities across its AI products."
    },
    {
      "id": 2,
      "text": "Several months later, Google integrated Gemini 2.5 into Google Workspace,"
    },
    {
      "id": 3,
      "text": "allowing users to generate documents, summarize emails, and analyze spreadsheets."
    }
  ],
  "items": [
    {
      "id": "relation_0",
      "label": "Gemini 2.5 --integrated into--> Google Workspace",
      "edu_ids": [
        2
      ],
      "edu_texts": [
        "Several months later, Google integrated Gemini 2.5 into Google Workspace,"
      ]
    },
    {
      "id": "relation_1",
      "label": "Google --integrated--> Gemini 2.5",
      "edu_ids": [
        2
      ],
      "edu_texts": [
        "Several months later, Google integrated Gemini 2.5 into Google Workspace,"
      ]
    },
    {
      "id": "relation_2",
      